# Fixed ensemble diversity screen v010

This notebook evaluates exactly the two candidates declared in `research/ENSEMBLE_HYPOTHESES_v010.md` before calculation: deep1/2 plus original compact/curve/memory each1/6; deep1/2 plus original compact/curve/memory/attention each1/8. There is no roster or weight search.

Inputs are existing verified native A predictions and approved original A validation labels. Prediction, model, record, source, approved-data and row-ID hashes are verified against retained records and the403-check independent v008 audit. No model is loaded or executed, no feature is fitted, no D split is constructed or scored, and no hidden score is queried. Pressure labels are used only for metrics and error-correlation diagnostics.

Set `PA3_RUN_DIVERSITY_V010=1` explicitly to run once and write `outputs/diversity_screen_v010/results.json`. Default execution only reviews an existing report. The code hash covers code cells rather than changing execution outputs. The original A screen is repeatedly used research evidence. Qualification here remains provisional until v009 and these fixed comparisons finish; this notebook performs no promotion, confirmation, final fitting or submission.


In [1]:
import hashlib
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

V010_RUN = os.environ.get('PA3_RUN_DIVERSITY_V010', '0') == '1'
V010_ROOT = Path.cwd().resolve()
V010_NOTEBOOK = Path('research/diversity_screen_v010.ipynb')
V010_DESTINATION = Path('outputs/diversity_screen_v010/results.json')
V010_HYPOTHESES_SHA256 = '587249f7547b9034c12377e19459846bbb1a688f2b96e87c715971b0c4ee7dc9'
V010_AUDIT_SHA256 = '9e92d039d990610a2a440be9db3c88e77704e0404e4106cdfc86034cc1f62f91'
V010_MODELS = ('original_deep', 'original_compact', 'curve_consistency',
    'learned_memory', 'residual_attention')
V010_CANDIDATES = (
    ('deep_half_compact_curve_memory',
        (('original_deep', 1/2), ('original_compact', 1/6),
         ('curve_consistency', 1/6), ('learned_memory', 1/6))),
    ('deep_half_compact_curve_memory_attention',
        (('original_deep', 1/2), ('original_compact', 1/8),
         ('curve_consistency', 1/8), ('learned_memory', 1/8),
         ('residual_attention', 1/8))))


def v010_file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def v010_array_sha(values):
    return hashlib.sha256(np.ascontiguousarray(values).tobytes()).hexdigest()


def v010_code_sha(path):
    notebook = json.loads(Path(path).read_text())
    source = '\n'.join(''.join(cell.get('source', [])) for cell in notebook['cells']
        if cell.get('cell_type') == 'code')
    return hashlib.sha256(source.encode()).hexdigest()


def v010_json_sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def v010_local(root, saved_path):
    value = str(saved_path)
    marker = '/'+root.name+'/'
    path = (root/value.split(marker, 1)[1] if marker in value else root/value).resolve()
    if not path.is_relative_to(root) or not path.is_file():
        raise ValueError('Missing artifact or path outside repository: '+value)
    return path


def v010_metrics(prediction, truth, counts):
    error = np.abs(prediction-truth)
    starts = np.r_[0, np.cumsum(counts)[:-1]]
    return dict(mae=float(error.mean()), median_absolute_error=float(np.median(error)),
        p90_absolute_error=float(np.quantile(error, .9)),
        median_breath_mae=float(np.median(np.add.reduceat(error, starts)/counts)))


def v010_paired_interval(new_error, reference_error, counts):
    """Paired whole-breath resampling, retaining point-weighted native MAE."""
    sums = np.add.reduceat(new_error-reference_error, np.r_[0, np.cumsum(counts)[:-1]])
    rng = np.random.default_rng(20261201)
    draws = []
    for _ in range(10):
        selected = rng.integers(0, len(counts), size=(100, len(counts)))
        draws.extend((sums[selected].sum(axis=1)/counts[selected].sum(axis=1)).tolist())
    low, high = np.quantile(draws, [.025, .975])
    return dict(difference=float((new_error-reference_error).mean()),
        bootstrap_95_low=float(low), bootstrap_95_high=float(high), seed=20261201,
        repeats=1000, unit='whole breath', point_count_weighted=True,
        direction='candidate minus original fixed pair; negative favors candidate')


def v010_run_fixed_screen(root=V010_ROOT):
    """Exactly two frozen saved-prediction calculations; no model/feature execution."""
    if os.environ.get('PA3_RUN_DIVERSITY_V010', '0') != '1':
        raise RuntimeError('Explicit PA3_RUN_DIVERSITY_V010=1 is required')
    root = Path(root).resolve()
    destination = root/V010_DESTINATION
    if destination.exists():
        raise FileExistsError('Existing completed screen is preserved: '+str(destination))
    source_before = v010_code_sha(root/V010_NOTEBOOK)
    checks, inputs = [], {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks.append(name)
    def read(path):
        return json.loads(Path(path).read_text())
    def verify(path, expected=None):
        path = Path(path)
        digest = v010_file_sha(path)
        if expected is not None:
            check('file_hash:'+str(path.relative_to(root)), digest == expected)
        inputs[str(path.relative_to(root))] = dict(file_sha256=digest)
        return digest

    verify(root/'research/ENSEMBLE_HYPOTHESES_v010.md', V010_HYPOTHESES_SHA256)
    verify(root/'research/screen_review_v008.json', V010_AUDIT_SHA256)
    audit = read(root/'research/screen_review_v008.json')
    check('independent_v008_audit_complete_passed403', audit['passed']
        and audit['status'] == 'passed' and audit['check_count'] == 403
        and not audit['failed_checks'] and not audit['missing_artifacts']
        and all(item['passed'] for item in audit['checks']))
    verify(root/'outputs/controlled_experiments_v008/review/audit_script.py',
        audit['source_script_sha256'])
    out, base = root/'outputs/controlled_experiments_v008', root/'outputs/improvements_v005/sequence'
    design, base_design = read(out/'design.json'), read(base/'design.json')
    verify(out/'design.json'); verify(base/'design.json')
    approval = read(root/'research/authorization_v008.json')
    verify(root/'research/authorization_v008.json', design['authorization_sha256'])
    check('standing_controlled_approval_and_approved_data',
        approval['standing_approval_for_controlled_iterations'] and approval['only_assignment_approved_data'])
    check('same_approved_training_data', design['train_sha256'] == base_design['train_sha256'])
    verify(root/'data/train.csv', design['train_sha256'])
    source_manifest = {}
    for filename, expected in design['component_hashes'].items():
        path = root/'research'/filename
        actual = v010_code_sha(path)
        check('component_code_hash:'+filename, actual == expected)
        source_manifest[str(path.relative_to(root))] = dict(code_sha256=actual, file_sha256=verify(path))
    for filename, expected in [('controlled_experiments_v008.ipynb', design['source_sha256']),
            ('iterative_improvements_v005.ipynb', base_design['source_code_sha256'])]:
        actual = v010_code_sha(root/filename)
        check('model_orchestration_code_hash:'+filename, actual == expected)
        source_manifest[filename] = dict(code_sha256=actual, file_sha256=verify(root/filename))

    # Match the auditor: approved labels are parsed as float32, then selected A
    # inspirational labels alone are converted to float64 for native arithmetic.
    frame = pd.read_csv(root/'data/train.csv', usecols=['id', 'breath_id', 'u_out', 'pressure'],
        dtype=dict(id='int64', breath_id='int32', u_out='int8', pressure='float32'))
    check('approved_complete_row_count', len(frame) == 67905*80)
    blocks = frame.breath_id.to_numpy().reshape(-1, 80)
    breath_ids = blocks[:, 0]
    check('complete_unique_breath_blocks', np.all(blocks == breath_ids[:, None])
        and len(np.unique(breath_ids)) == 67905 and frame.id.is_unique)
    verify(base/'split_breath_ids.npz')
    # Access original A groups only; do not construct or inspect D split data.
    with np.load(base/'split_breath_ids.npz', allow_pickle=False) as split:
        fitting = np.sort(pd.Index(breath_ids).get_indexer(split['A_fit']))
        validation = np.sort(pd.Index(breath_ids).get_indexer(split['A_validation']))
    check('original_A_counts_disjoint', len(fitting) == 61114 and len(validation) == 6791
        and (fitting >= 0).all() and (validation >= 0).all()
        and len(np.unique(fitting)) == len(fitting) and len(np.unique(validation)) == len(validation)
        and not np.intersect1d(fitting, validation).size)
    for key, indices in [('A_fit', fitting), ('A_validation', validation)]:
        check('original_group_hash:'+key,
            v010_array_sha(breath_ids[indices]) == base_design['group_hashes'][key]
            == design['plan']['group_hashes'][key])
    valve = frame.u_out.to_numpy().reshape(-1, 80)[validation]
    check('known_binary_inspiratory_mask', np.isin(valve, (0, 1)).all()
        and (valve[:, 0] == 0).all() and (np.diff(valve, axis=1) >= 0).all())
    mask = valve == 0
    counts = mask.sum(axis=1)
    truth32 = frame.pressure.to_numpy().reshape(-1, 80)[validation][mask]
    truth = truth32.astype(np.float64)
    row_ids = frame.id.to_numpy().reshape(-1, 80)[validation][mask]
    check('scored_truth_shape_dtype_finite', truth32.dtype == np.float32
        and truth.shape == (206098,) and np.isfinite(truth).all() and (counts > 0).all())
    for path in [base/'A_validation_row_ids.npy', out/'A_row_ids.npy']:
        verify(path)
        check('exact_A_row_alignment:'+str(path.relative_to(root)),
            np.array_equal(row_ids, np.load(path, allow_pickle=False)))
    row_hash = v010_array_sha(row_ids)

    paths = dict(original_deep=base/'record_A_capacity_deep_bilstm.json',
        original_compact=base/'record_A_capacity_bilstm128.json',
        **{name: out/('record_A_'+name+'.json') for name in V010_MODELS[2:]})
    predictions, members = {}, {}
    for name in V010_MODELS:
        record_path = paths[name]
        record = read(record_path)
        verify(record_path, audit['verified_file_sha256'][str(record_path.relative_to(root))])
        if name in V010_MODELS[:2]:
            check('baseline_record_in_frozen_v008_design:'+name,
                v010_file_sha(record_path) == design['baseline_records'][record['name']])
        else:
            check('candidate_record_A_row_hash:'+name, record['row_ids_sha256'] == row_hash)
            check('candidate_record_frozen_design:'+name, record['design_sha256'] == v010_json_sha(design)
                and record['recipe']['component_hashes'] == design['component_hashes'])
        model_path = v010_local(root, record['checkpoint_path'])
        model_hash = verify(model_path, record['checkpoint_sha256'])
        check('model_hash_matches_independent_audit:'+name,
            model_hash == audit['verified_file_sha256'][str(model_path.relative_to(root))])
        prediction_path = v010_local(root, record['prediction_file'])
        prediction_file_hash = verify(prediction_path)
        prediction = np.load(prediction_path, allow_pickle=False)
        check('saved_native_prediction_hash:'+name,
            v010_array_sha(prediction) == record['prediction_sha256'])
        check('saved_native_prediction_shape_dtype_finite:'+name,
            prediction.shape == truth.shape and prediction.dtype == np.float64 and np.isfinite(prediction).all())
        predictions[name] = prediction
        individual = v010_metrics(prediction, truth, counts)
        check('individual_MAE_matches_verified_record:'+name,
            np.isclose(individual['mae'], record['validation_mae'], rtol=0., atol=1e-12))
        members[name] = dict(record_file=str(record_path.relative_to(root)),
            record_file_sha256=v010_file_sha(record_path), checkpoint_file=str(model_path.relative_to(root)),
            checkpoint_file_sha256=model_hash, prediction_file=str(prediction_path.relative_to(root)),
            prediction_file_sha256=prediction_file_hash, prediction_raw_array_sha256=record['prediction_sha256'],
            prediction_shape=list(prediction.shape), prediction_dtype=prediction.dtype.str,
            row_ids_raw_array_sha256=row_hash, best_epoch=record['best_epoch'],
            selected_lr_schedule=record['selected_lr_schedule'], metrics=individual)

    reference = .5*(predictions['original_deep']+predictions['original_compact'])
    reference_metrics = v010_metrics(reference, truth, counts)
    check('original_pair_exact_native_MAE', np.isclose(reference_metrics['mae'],
        .19656260982217663, rtol=0., atol=1e-12))
    check('original_pair_matches_independent_audit', np.isclose(reference_metrics['mae'],
        audit['results']['reference']['mae'], rtol=0., atol=1e-12))
    reference_error = np.abs(reference-truth)
    results = []
    check('exactly_two_predeclared_candidates', len(V010_CANDIDATES) == 2)
    for name, weights in V010_CANDIDATES:
        check('fixed_weights_sum_one:'+name, np.isclose(sum(weight for _, weight in weights),
            1., rtol=0., atol=1e-15))
        prediction = sum(weight*predictions[member] for member, weight in weights)
        metric = v010_metrics(prediction, truth, counts)
        interval = v010_paired_interval(np.abs(prediction-truth), reference_error, counts)
        gain = reference_metrics['mae']-metric['mae']
        qualifies = bool(gain >= .003 and interval['bootstrap_95_high'] < 0.)
        results.append(dict(name=name, weights=dict(weights), metrics=metric,
            gain_vs_original_pair=float(gain), interval=interval,
            qualifies_provisionally=qualifies, prediction_raw_array_sha256=v010_array_sha(prediction)))
    # Diagnostic only: these correlations do not choose weights or membership.
    signed_errors = np.stack([predictions[name]-truth for name in V010_MODELS])
    correlations = np.corrcoef(signed_errors)
    check('signed_error_correlation_matrix_finite', np.isfinite(correlations).all()
        and np.allclose(np.diag(correlations), 1., rtol=0., atol=1e-12))
    source_after = v010_code_sha(root/V010_NOTEBOOK)
    check('notebook_code_hash_unchanged_during_screen', source_before == source_after)
    report = dict(schema='fixed_diversity_screen_v010', created_at_utc=datetime.now(timezone.utc).isoformat(),
        status='complete_provisional_no_promotion', candidate_count=2, weight_or_roster_search=False,
        hypotheses_file='research/ENSEMBLE_HYPOTHESES_v010.md', hypotheses_sha256=V010_HYPOTHESES_SHA256,
        code_hash_before_run=source_before, code_hash_after_run=source_after,
        notebook_code_hash_unchanged=True, original_A_validation_breaths=6791,
        inspiratory_points=int(len(truth)), row_ids_raw_array_sha256=row_hash,
        native_scored_truth_raw_array_sha256=v010_array_sha(truth), label_parse_dtype='float32',
        metric_arithmetic_dtype='float64', native_point_weighted_inspiratory_MAE=True,
        reference=dict(weights=dict(original_deep=.5, original_compact=.5), metrics=reference_metrics),
        results=results, signed_error_correlations=dict(model_order=list(V010_MODELS),
            pearson=correlations.tolist(), diagnostic_only=True), members=members,
        source_manifest=source_manifest, verified_inputs=inputs, checks_passed=len(checks), checks=checks,
        independent_v008_audit=dict(path='research/screen_review_v008.json',
            file_sha256=V010_AUDIT_SHA256, passed_checks=403),
        gate=dict(minimum_gain=.003, whole_breath_bootstrap_upper95_below_zero=True,
            seed=20261201, repeats=1000), selected_for_confirmation=None,
        selection_status='deferred_until_v009_and_fixed_screens_complete',
        training_performed=False, model_loading_performed=False, inference_performed=False,
        D_split_accessed_or_constructed=False, confirmation_performed=False,
        final_fit_performed=False, new_submission_performed=False, official_best_preserved=.18550,
        limitations=['Original A is repeatedly used research/model-selection evidence, not independent confirmation.',
            'Qualification is provisional; no D recipe is selected or promoted by this screen.',
            'Saved checkpoint hashes are verified without loading models or regenerating predictions.'],
        runtime=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__))
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name('results.json.tmp')
    temporary.write_text(json.dumps(report, indent=2, allow_nan=False)+'\n')
    os.replace(temporary, destination)
    return report


In [2]:
def v010_review_table(report):
    rows = [dict(name='original_compact_deep_pair',
        **report['reference']['metrics'], gain=0., bootstrap_95_low=None,
        bootstrap_95_high=None, qualifies_provisionally=False)]
    for result in report['results']:
        rows.append(dict(name=result['name'], **result['metrics'],
            gain=result['gain_vs_original_pair'],
            bootstrap_95_low=result['interval']['bootstrap_95_low'],
            bootstrap_95_high=result['interval']['bootstrap_95_high'],
            qualifies_provisionally=result['qualifies_provisionally']))
    return pd.DataFrame(rows)

if V010_RUN:
    V010_REPORT = v010_run_fixed_screen()
    print(v010_review_table(V010_REPORT).to_string(index=False, float_format=lambda value: f'{value:.9f}'))
    print('Status:', V010_REPORT['status'])
    print('Source code SHA256:', V010_REPORT['code_hash_before_run'])
    print('Verified checks:', V010_REPORT['checks_passed'])
elif (V010_ROOT/V010_DESTINATION).is_file():
    V010_REPORT = json.loads((V010_ROOT/V010_DESTINATION).read_text())
    print(v010_review_table(V010_REPORT).to_string(index=False, float_format=lambda value: f'{value:.9f}'))
    print('Review only; source code SHA256:', V010_REPORT['code_hash_before_run'])
else:
    print('Review only. Explicit PA3_RUN_DIVERSITY_V010=1 is required to run the two fixed candidates.')


                                    name         mae  median_absolute_error  p90_absolute_error  median_breath_mae        gain  bootstrap_95_low  bootstrap_95_high  qualifies_provisionally
              original_compact_deep_pair 0.196562610            0.100945950         0.428331375        0.118814320 0.000000000               NaN                NaN                    False
          deep_half_compact_curve_memory 0.192230255            0.098696550         0.425293334        0.116131163 0.004332355      -0.005840202       -0.003035467                     True
deep_half_compact_curve_memory_attention 0.192303462            0.099206746         0.425863743        0.115429066 0.004259148      -0.005772481       -0.002932763                     True
Status: complete_provisional_no_promotion
Source code SHA256: 6762d765e70aee25eb722de1098ccba111539bedc29f55dd16bf2ae6aee8b397
Verified checks: 69
